## A simple Tephra2 workflow
Tephra2 is used to simulate tephra fallout deposits associated with explosive volcanic eruptions. In the forward solution, eruptions source parameters (ESPs) are specified, a wind field is specified, and the map locations where tephra accumulation is calculated are specified. Tephra2 produces estimates of mass loading or grainsize distribution at these map locations based on the ESPs and wind data sets. 

Explore this workflow if you are interested in:
1. Simulating an explosive eruption and generating an isomass map with Tephra2
2. Learning about ESPs and their impact on tephra accumulation
3. Creating input files for Tephra2
4. Executing compiled codes from the notebook
5. Manipulating model output with pandas and making simple contour plots.


### The workflow
The steps in this workflow are:
1. Build a configuration file for Tephra2 consisting of eruption source parameters and save this file.
2. Build a grid file for Tephra2 consisting of the points where tephra accumulation will be modeled, and save this file.
3. Build a wind file for Tephra2 consisting of wind velocity as a function of height, and save this file
4. Run Tephra2
5. Parse the Tephra2 output to obtain isomass information
6. Plot an isomass map from the parsed Tephra2 output.

### A few notes for simplicity
Unless explicitly written at the top of the cell, assume each cell needs to simply be run without alteration.


### Some Tephra2 references and examples

Bonadonna, C., Connor, C.B., Houghton, B.F., Connor, L., Byrne, M., Laing, A. and Hincks, T.K., 2005. Probabilistic modeling of tephra dispersal: Hazard assessment of a multiphase rhyolitic eruption at Tarawera, New Zealand. Journal of Geophysical Research: Solid Earth, 110(B3). <a href="https://agupubs.onlinelibrary.wiley.com/doi/pdf/10.1029/2003JB002896">Link</a>


run <a href="https://gscommunitycodes.usf.edu/geoscicommunitycodes/public/tephra2/tephra2.php">Tephra2</a> interactively on the web

### Required installed libraries for this workflow

In [1]:
import os  # needed to run tephra2 externally
import pandas as pd #used to manipulate tephra2 output in this notebook
import matplotlib.pyplot as plt # used to plot the isopach map
import matplotlib.tri as tri # used to plot the isopach map
import numpy as np
import cdsapi #used to pull reanalysis data
import netCDF4 #used to process downloaded netCDF format data
import subprocess #for proper program running
import utm
import sys
sys.path.insert(0, "/home/jovyan/shared/Libraries/")
import victor
import cartopy.crs as ccrs
from cartopy.io.img_tiles import Stamen
import matplotlib.pyplot as plt
import rioxarray as rxr

### Enter the coordinates of your volcano below

In [2]:
#Coordinate location
vent_latitude, vent_longitude = 45.22, -121.44 #44.1, -121.77

### The eruption parameter file
The Tephra2 configuration file contains all the ESPs information required to run the model.

The Tephra2 code reads the configuration file and searchs for values associated with keywords. Change the values to change that eruption source parameter. Do not change the keyword.

The keywords, definitions, units, and example values are as follows:
1.  __Done Automatically__ VENT_EASTING: this is the easting (x) coordinate of the volcanic vent that produces the tephra. The value can be a local grid or an actual UTM grid value. Units are meters. Example: 450000 or 0
2. __Done Automatically__ VENT_NORTHING: this is the northing (y) coordinate of the volcanic vent that produces the tephra. The value can be a local grid or an actual UTM grid value. Units are meters. Example: 4500000 or 0
3. __Done Automatically__ VENT_ELEVATION: the elevation of the volcanic vent. Units are meters above sealevel. Example 1500

In [3]:
name = victor.download_dem(vent_latitude+1, vent_latitude-1, vent_longitude+1, vent_longitude-1, "tiff", "SRTMGL3", filename="tephra2.tiff", api_key='73d3f5f5000048606a3c15ff635bde63')

File already exists with name DEM_46N_120W_SRTMGL3_06082026.geotiff, exiting.


In [4]:
converted = utm.from_latlon(vent_latitude,vent_longitude)

vent_easting = converted[0]

vent_northing = converted[1]

dem = rxr.open_rasterio(name)
vent_elevation = dem.sel(x=vent_longitude,y=vent_latitude, method="nearest").values[0]

#### Input These Below
4. PLUME_HEIGHT: the height of the top of the volcanic plume. Units are meters above sealevel. Example: 12000
5. ERUPTION_MASS: the total mass of tephra produced by the eruption. Units are kg. Examples: 1e12, 10000000.
6. MEDIAN_GRAINSIZE: the median of the total grainsize distribution of the eruption. Tephra2 assumes the total grainsize distribution is normally distributed. Units are $\phi$. Examples: 0, -1, 1
7. STD_GRAINSIZE: the standard deviation of the total grainsize distribution. Units are $\phi$. Examples: 1, 2

These seven values are required for accurate modeling of tephra. The following eleven parameters (8-19) may be left empty in place of default trusted values if data is unavailable.

In [5]:
#plume_height = 12000

#eruption_mass = 3e8

median_grain = 1

std_grain = 1

### These parameters allow for additional specificity, but can be left blank to use defaults.

8. FALL_TIME_THRESHOLD: the fall time threshold is the time at which diffusion laws switch in Tephra2, as different particle sizes have differnt diffusion in the atmosphere. Units are seconds. Examples: 1000, 1e6
9. Plume model: The plume model descibed how tephra mass is distributed in the eruption column from the vent to the total height of the volcanic plume. Options are (1) using a uniform distribution, or (2) using a beta distribution. Dimensionless.
10. PART_STEPS: Settling velocity is a function of grainsize. Tephra2 divides the total grainsize distribution into particles steps and calculates the settling velocity for each particle step. Dimensionless. Examples: 100, 200
11. DIFFUSION_COEFFICIENT: The spreading of the plume perpendicular to the major axis of dispersion is controled by the diffusion coefficient. Units are meters squared per second. Examples: 300, 3000, 300000
12. LITHIC_DENSITY: The density of small particles in the tephra. Units are kg/m^3. Examples are: 2600, 2800
13. COL_STEPS: Tephra2 discretizes the eruption column. The number of column steps is how many layers occur in the plume from the vent to the total hieght of the plume. Units are meters. Examples: 100, 200
14. MAX_GRAINSIZE: the maximum size of particles in the modeled plume. These particles must be small enough to be convected by the plume. Units are $\phi$. Examples: -4, -3.5
15. ALPHA: the alpha ($\alpha$) parameter in the Beta distribution to control the mass of tephra released as a function of height in the erupiton column (PLUME_MODEL 2). Dimensionless. Example 1, 2
16. BETA: the beta ($\beta$) parameter in the Beta distribution to control the mass of tephra released as a function of height in the erupiton column (PLUME_MODEL 2). Dimensionless. Example 1, 2
17. MIN_GRAINSIZE: the minimum size of particles in the modeled plume.  Units are $\phi$. Examples: 4, 3.5 (note: aggregation is not considered in this model)
18. PUMICE_DENSITY: The The density of large particles in the tephra. Units are kg/m^3. Examples are: 1000, 800
19. EDDY_CONST: The turbulent eddy constant for the atmosphere. Units are meters sqaured per second. Example: 0.04

In [6]:
falltime_thresh = 0

plume_model = 0

part_steps = 0

#diffusion_coef = 100000

lithic_density = 0

col_steps = 0

max_grain = 0

alpha = 0

beta = 0

min_grain = 0

pumice_density = 0

eddy_const = 0

### The grid file
The grid file is a simple list of locations and elevations where tephra accumulation will be calculated. Note that the elevations of all points in the grid file must be the same. This is because Tephra2 using an integral solution to the advection-diffusion equation to simplify the modeling. That is, the tephra accumulates on a flat topography.

Alter the code in the following cell to create your own grid file. Below, please enter a list of longitudes and latitudes of measurement points, as well as an altitude in meters for the given points. Enter the values as a matrix as shown below.

In [89]:
#INPUT GRID PARAMETERS HERE

#Specify easting of volcano
vol_easting = round(converted[0])
#Specify northing of volcano
vol_northing = round(converted[1])

#Specify distance around volcano (in meters, default 50km or 50,000 m)
grid_radius = []

#Specify spacing in grid (in meters, default 1km or 1,000 m)
grid_spacing = []

#Specify elevation of points, constant due to integral solution used (in meters, default 1,500 m)
elevation = []

In [90]:
#in this example, the grid cells are 1 km x 1km, the volcano is located
# at 0,0 (km) and the grid extends 100 km from the volcano (rectangle)

# the data in the grid file always have the form
# easting northing elevation

min_easting = vol_easting-100000 if not grid_radius else vol_easting-grid_radius
max_easting = vol_easting+100000 if not grid_radius else vol_easting+grid_radius
min_northing = vol_northing-100000 if not grid_radius else vol_northing-grid_radius
max_northing = vol_northing+100000 if not grid_radius else vol_northing+grid_radius
grid_size = 1000 if not grid_spacing else grid_spacing
elevation = 1500 if not elevation else elevation

In [ ]:
#For South Sister: locations = [[44.0569637,-121.3320308],[44.2637621,-121.1753231],[43.8741362,-121.4465950]]

#For Hood:
locations = [45.329563, -121.911191],[45.519839, -121.596742], [45.1808, -121.4509] #Rhododendron, Parkdale, Govt. Camp
elevation = 1000

locations = np.array(locations)
loced = utm.from_latlon(locations[:,0],locations[:,1])

In [ ]:
with open("volcano_cone.grid", "w") as output_file:
    for i in range (min_easting, max_easting, grid_size):
        for j in range (min_northing, max_northing, grid_size):
            if (i != vol_easting and j != vol_northing):
                print(i, j, elevation, file = output_file)
    for i in range(locations.shape[0]):
        print(f"{round(loced[0][i])} {round(loced[1][i])} {elevation}", file=output_file)


### The wind file
Tephra2 requires that the wind velcocity be specifed as a function of height above the ground surface. The code in the following cells pulls wind reanalysis data and manipulates it to give Tephra2 compatible data. Change the values to change the wind velocity as a function of height in your simulation. Any number of wind heights may be specified by adding another pressure value. 

In [ ]:
#Specify month(s) (in numerical string format, e.g. "01", "11")
months = ["10"]

#Specify year(s) (in numerical string format, e.g. "1999")
years = ["2024"]

#Specify day(s) (in numerical string format, e.g. "01", "11")
days = ["23"]

#Specify hour(s) in 24 hour string format (00:00 - 23:00)
hours = ['17:00']

#Specify pressure values to generate wind field around volcano

#valid values are 
# ['1', '2', '3', '5', '7', '10', '20', '30', '50','70', '100', '125','150', '175', '200','225', '250', '300', '350', '400'
# '450','500', '550', '600','650', '700', '750','775', '800', '825','850', '875', '900','925', '950', '975','1000']
pressures = ['1', '2', '3', '5', '7', '10', '20', '30', '50', '70', 
             '100', '125', '150', '175', '200', '225', '250', '300', 
             '350', '400', '450', '500', '550', '600', '650', '700', 
             '750', '775', '800', '825', '850', '875', '900', '925', 
             '950', '975', '1000']

#Specify name of output file, in netCDF format (optional, default is download.nc)
file_name = []

In [ ]:
north = round(vent_latitude*4)/4
south = north +.1
east = round(vent_longitude*4)/4
west = east - .1

In [ ]:
wind_data = cdsapi.Client()
dataset = "reanalysis-era5-pressure-levels"
request = {
        "product_type": ["reanalysis"],
        "data_format": "netcdf",
        "variable": [
            "geopotential", "u_component_of_wind", "v_component_of_wind",
        ],
        "pressure_level": pressures,
        "year": years,
        "month": months,
        "day": days,
        "time": hours,
        "download_format": "unarchived",
        "area": [
            north, west, south, east
        ],
    }
wind_data.retrieve(dataset, request, "download.nc")

In [ ]:
wind = netCDF4.Dataset("download.nc")

In [ ]:
uwnd = wind["u"][0,:,0,0]
vwnd = wind["v"][0,:,0,0]

speed = np.sqrt(vwnd**2 + uwnd**2)
direction = -180/np.pi *np.arctan(vwnd/uwnd)
for d in range(len(direction)):
    if uwnd[d] > 0:
        direction[d] += 90
    else:
        direction[d] += 270

hgt = (wind["z"][0,:,0,0])/9.80665
speed = speed[::-1]
direction = direction[::-1]
hgt = hgt[::-1]

In [ ]:
row2 = []
for l in range(wind["pressure_level"].shape[0]):
    row2.append(str(hgt[l]) + ' ' + str(speed[l]) + ' ' + str(direction[l]))

# Format of data islevel (masl) speed (m/s) direction that wind is blowing toward (degrees)
wind_file = open("my_wind.dat", "w")
for element in row2:
    wind_file.write("%s\n" % element)
wind_file.close()

### Running Tephra2
The following script runs Tephra2. The files:

1. configuration file
2. grid file
3. wind file

must be created first. See examples in previous cells in this notebook.
The command creates the tephra2.out file, in which output is stored.



In [ ]:
import numpy as np
import pandas as pd

ph_start = 1000
ph_end = 24000
em_start = 1e9
em_end = 1e12
dc_start = 1e3
dc_end = 1e5

plume_heights = np.linspace(ph_start, ph_end, 2, dtype=int)
eruption_masses = np.linspace(em_start, em_end, 2, dtype=int)
diffusion_coefs = np.linspace(dc_start, dc_end, 2, dtype=int)

# Create aggregated output file with header
agg_filename = "tephra2_aggregated.csv"
with open(agg_filename, 'w') as f:
    f.write("plume_height,eruption_mass,diffusion_coef,easting,northing,mass_kg_m2\n")

for plume_height in plume_heights:
    for eruption_mass in eruption_masses:
        for diffusion_coef in diffusion_coefs:
            row = []
            print(plume_height, eruption_mass, diffusion_coef)

            row.append('VENT_EASTING ' + str(vent_easting))
            row.append('VENT_NORTHING ' + str(vent_northing))
            row.append('VENT_ELEVATION ' + str(vent_elevation))
            row.append('PLUME_HEIGHT ' + str(plume_height))
            row.append('ERUPTION_MASS ' + str(eruption_mass))
            row.append('MEDIAN_GRAINSIZE ' + str(median_grain))
            row.append('STD_GRAINSIZE ' + str(std_grain))
            row.append("".join(('FALL_TIME_THRESHOLD ', str(1000) if not falltime_thresh else str(falltime_thresh))))
            row.append("".join(('PLUME_MODEL ', str(2) if not plume_model else str(plume_model))))
            row.append("".join(('PART_STEPS ', str(100) if not part_steps else str(part_steps))))
            row.append("".join(('LITHIC_DENSITY ', str(2600.0) if not lithic_density else str(lithic_density))))
            row.append("".join(('COL_STEPS ', str(200) if not col_steps else str(col_steps))))
            row.append("".join(('MAX_GRAINSIZE ', str(-4) if not max_grain else str(max_grain))))
            row.append("".join(('ALPHA ', str(1) if not alpha else str(alpha))))
            row.append("".join(('BETA ', str(1) if not beta else str(beta))))
            row.append("".join(('MIN_GRAINSIZE ', str(4) if not min_grain else str(min_grain))))
            row.append("".join(('PUMICE_DENSITY ', str(1000.0) if not pumice_density else str(pumice_density))))
            row.append("".join(('EDDY_CONST ', str(0.04) if not eddy_const else str(eddy_const))))
            row.append('DIFFUSION_COEFFICIENT ' + str(diffusion_coef))

            config_file = open("my_esps.conf", "w")
            for element in row:
                config_file.write(element + "\n")
            config_file.close()

            # Run tephra2 to a temporary csv
            csv_filename = f"tephra2_temp.csv"
            result = subprocess.run(
                f'/home/jovyan/tephra2/tephra2_2020 my_esps.conf volcano_cone.grid my_wind.dat > {csv_filename} 2>/dev/null',
                shell=True)
            os.remove("my_esps.conf")

            if result.returncode != 0:
                print(result.stderr.decode())
                continue
            
            # ADDED — read the temp output file
            tephra_out = pd.read_csv(csv_filename, sep='\s+')


            # Read output and append to aggregated file
            tephra_out = tephra_out.rename(columns={'#EAST': 'easting', 'NORTH': 'northing', 'Kg/m^2': 'mass_kg_m2'})
            tephra_out = pd.concat([
                pd.DataFrame({
                    'plume_height': plume_height,
                    'eruption_mass': eruption_mass,
                    'diffusion_coef': diffusion_coef
                }, index=tephra_out.index),
                tephra_out[['easting', 'northing', 'mass_kg_m2']]
            ], axis=1)
            tephra_out.to_csv(agg_filename, mode='a', header=False, index=False)

            os.remove(csv_filename)

SINGLE ISOMASS MAP

The single isomass map shows the result of one specific parameter combination. You need to choose:

A specific plume height (e.g. 12,000 m)
A specific eruption mass (e.g. 5e10 kg)
A specific diffusion coefficient (e.g. 5e4 m²/s)

And the map will show the ash dispersal pattern for that one scenario. This is why the single-run cell needs to be separate from the sweep — it is a deliberate choice of which scenario to visualize, not an iteration.

Later, when the analysis notebook is built, you will be able to query the aggregated CSV to find the most interesting scenarios to visualize — for example the parameter combination that produced the maximum ash thickness — and then feed those values into this single-run cell to generate a targeted isomass map.


In [ ]:
# Single Tephra2 run for isomass map
# Adjust these values to visualize a specific scenario
single_plume_height    = 12000   # meters above sea level (mid-range)
single_eruption_mass   = 5e10    # kg (mid-range)
single_diffusion_coef  = 5e4     # m²/s (mid-range)

# Build config file
row_single = []
row_single.append('VENT_EASTING '          + str(vent_easting))
row_single.append('VENT_NORTHING '         + str(vent_northing))
row_single.append('VENT_ELEVATION '        + str(vent_elevation))
row_single.append('PLUME_HEIGHT '          + str(single_plume_height))
row_single.append('ERUPTION_MASS '         + str(single_eruption_mass))
row_single.append('MEDIAN_GRAINSIZE '      + str(median_grain))
row_single.append('STD_GRAINSIZE '         + str(std_grain))
row_single.append("".join(('FALL_TIME_THRESHOLD ', str(1000)   if not falltime_thresh else str(falltime_thresh))))
row_single.append("".join(('PLUME_MODEL ',         str(2)      if not plume_model     else str(plume_model))))
row_single.append("".join(('PART_STEPS ',          str(100)    if not part_steps      else str(part_steps))))
row_single.append("".join(('LITHIC_DENSITY ',      str(2600.0) if not lithic_density  else str(lithic_density))))
row_single.append("".join(('COL_STEPS ',           str(200)    if not col_steps       else str(col_steps))))
row_single.append("".join(('MAX_GRAINSIZE ',       str(-4)     if not max_grain       else str(max_grain))))
row_single.append("".join(('ALPHA ',               str(1)      if not alpha           else str(alpha))))
row_single.append("".join(('BETA ',                str(1)      if not beta            else str(beta))))
row_single.append("".join(('MIN_GRAINSIZE ',       str(4)      if not min_grain       else str(min_grain))))
row_single.append("".join(('PUMICE_DENSITY ',      str(1000.0) if not pumice_density  else str(pumice_density))))
row_single.append("".join(('EDDY_CONST ',          str(0.04)   if not eddy_const      else str(eddy_const))))
row_single.append('DIFFUSION_COEFFICIENT '  + str(single_diffusion_coef))

config_file = open("my_esps.conf", "w")
for element in row_single:
    config_file.write(element + "\n")
config_file.close()

# Run Tephra2
result = subprocess.run(
    '/home/jovyan/tephra2/tephra2_2020 my_esps.conf volcano_cone.grid my_wind.dat > /home/jovyan/tephra2/tephra2.csv 2>/dev/null',
    shell=True)
os.remove("my_esps.conf")

if result.returncode != 0:
    print(result.stderr.decode())
else:
    print(f'Single run complete:')
    print(f'  Plume height:     {single_plume_height} m')
    print(f'  Eruption mass:    {single_eruption_mass} kg')
    print(f'  Diffusion coef:   {single_diffusion_coef} m²/s')
    print(f'Output written to /home/jovyan/tephra2/tephra2.csv')

### Manipulating Tephra2 output
The script in the following cell manipulates the tephra2 output. First, it reads the output file into a pandas dataframe. There is a lot of information in this dataframe. In this example, the easting, northing, and mass loading are extracted from the dataframe for each location in the grid file (where tephra accumulation was calculated). 

These extracted data are stored as lists, for ease in plotting.

In [ ]:
#use pandas to parse the output file
# in this case, ignore the granulometry and capture the tephra thickness data
# since the goal is to make an isomass map
tephra_out = pd.read_csv('/home/jovyan/tephra2/tephra2.csv', sep='\s+')

#extract rows from the pandas dataframe and convert to lists for easy plotting
easting = tephra_out['#EAST']
northing = tephra_out['NORTH']
mass = tephra_out['Kg/m^2']
lat, lon = utm.to_latlon(easting, northing, loced[2], loced[3])

length = -locations.shape[0]
lat, poi_lat = lat[0:length], lat[length:]
lon, poi_lon = lon[0:length], lon[length:]

mass, poi_mass = mass[0:length], mass[length:]

### Plotting the isomass map
The following script is an example of one way to plot a Tephra2 isomass data.

In [ ]:
#SET DISPLAY INPUTS HERE

#Specify levels in which solid countour lines should be generated
levels_solid = []

#Specify intermediate dashed lines for contour
levels_dashed = []

#specify file name/file type (jpg, png, pdf, svg)
output_image = "tephra_contour.png"

The following cell sets up the Cartopy projection and Colormap fill.

Key changes from the previous version:

plt.subplots replaced with fig, ax using subplot_kw={'projection': proj} — this is the Cartopy projection setup
All plt. calls replaced with ax. calls
transform=proj added to all plotting calls so Cartopy knows the data is in lat/lon
ax.gridlines added with labeled lat/lon grid
ax.set_extent replaces plt.xlim/ylim
dem.plot now passes ax=ax and transform=proj

Colormap Key changes:

scipy.griddata interpolates the scattered Tephra2 points onto a regular 500x500 grid
ax.contourf adds the YlOrRd colormap fill with LogNorm scaling behind the contour lines
Contour label colors changed from 'w' to 'k' so they are readable against the colored background

In [ ]:
import subprocess
subprocess.run('pip install matplotlib-scalebar --quiet', shell=True)

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.colors as mcolors
from scipy.interpolate import griddata

# plot a simple isomass map
proj = ccrs.PlateCarree()
fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={'projection': proj})

# DEM basemap
dem.plot(ax=ax, cmap="gray_r", add_colorbar=False, transform=proj)

# Interpolate scattered points onto a regular grid for contourf
grid_lon = np.linspace(lon.min(), lon.max(), 500)
grid_lat = np.linspace(lat.min(), lat.max(), 500)
glon, glat = np.meshgrid(grid_lon, grid_lat)
mass_interp = griddata((lon, lat), mass, (glon, glat), method='linear')

# Colormap fill — ADDED
norm = mcolors.LogNorm(vmin=1, vmax=1000)
cf = ax.contourf(glon, glat, mass_interp,
        levels=[1, 5, 10, 50, 100, 500, 1000],
        cmap='YlOrRd', norm=norm, alpha=0.65,
        transform=proj, extend='max')

# Colorbar — ADDED
cbar = plt.colorbar(cf, ax=ax, orientation='vertical', pad=0.08, shrink=0.6)
cbar.set_label('Ash Mass Loading (kg/m²)', fontsize=10)
cbar.set_ticks([1, 5, 10, 50, 100, 500, 1000])
cbar.set_ticklabels(['1', '5', '10', '50', '100', '500', '1000'])

# plot contours in kg/m^2 with solid lines
cntr2 = ax.tricontour(lon, lat, mass,
        levels=[1e-6, 1, 10, 100, 1000, 10000] if not levels_solid else levels_solid,
        colors=('k',), linestyles=('-',), linewidths=(1,), transform=proj)

# plot contours in kg/m^2 with dashed lines
cntr3 = ax.tricontour(lon, lat, mass,
        levels=[5, 50, 500] if not levels_dashed else levels_dashed,
        colors=('k',), linestyles=('dashed',), linewidths=(0.5,), transform=proj)

# label contours
ax.clabel(cntr2, fmt='%2.1d', colors='k', fontsize=8)
ax.clabel(cntr3, fmt='%2.1d', colors='k', fontsize=8)

# POI markers
poi_names = ['Parkdale', 'Rhododendron', 'Govt. Camp']
ax.scatter(poi_lon, poi_lat, marker="*", s=50, transform=proj, zorder=5)

for i in range(len(poi_lon)):
    ax.annotate(
        f"{poi_names[i]}\n{poi_mass.tolist()[i]:.2f} kg/m²",
        xy=(poi_lon.tolist()[i], poi_lat.tolist()[i]),
        xytext=(5, 5), textcoords='offset points',
        fontsize=7, transform=proj
    )

# plot the volcano location
ax.plot(vent_longitude, vent_latitude, "r^", markersize=8, transform=proj, zorder=6)

# Cartopy features
ax.gridlines(draw_labels=True, linewidth=0.4, color='gray', alpha=0.5, linestyle='--')


# Scale bar — manual implementation
scale_lon_start = lon.min() + 0.1
scale_lat = lat.min() + 0.05
scale_length_deg = 1.0  # approximately 80 km at this latitude

ax.plot([scale_lon_start, scale_lon_start + scale_length_deg], 
        [scale_lat, scale_lat], 
        'k-', linewidth=3, transform=proj)
ax.text(scale_lon_start + scale_length_deg / 2, scale_lat + 0.02, 
        '~80 km', ha='center', fontsize=8, transform=proj)

# North arrow — ADDED
ax.annotate('N', xy=(0.95, 0.95), xytext=(0.95, 0.88),
            xycoords='axes fraction',
            fontsize=14, fontweight='bold', ha='center',
            arrowprops=dict(arrowstyle='->', color='black', lw=2))

# map extent
ax.set_extent([lon.min(), lon.max(), lat.min(), lat.max()], crs=proj)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Tephra Dispersion $kg/m^2$")
plt.savefig(output_image)
plt.show()

Here is the GeoTIFF export cell.
This exports the same interpolated grid used for the colormap fill directly to a GeoTIFF with WGS84 (EPSG:4326) projection — the same CRS used in the Cartopy plot — so it will load and align correctly in QGIS

In [ ]:
import rioxarray
import xarray as xr
import numpy as np
from scipy.interpolate import griddata

# Interpolate mass onto a regular grid (same as plotting)
grid_lon = np.linspace(lon.min(), lon.max(), 500)
grid_lat = np.linspace(lat.min(), lat.max(), 500)
glon, glat = np.meshgrid(grid_lon, grid_lat)
mass_interp = griddata((lon, lat), mass, (glon, glat), method='linear')

# Fill NaN values with 0 (areas outside the dispersal grid)
mass_interp = np.where(np.isnan(mass_interp), 0, mass_interp)

# Create xarray DataArray with spatial coordinates
da = xr.DataArray(
    mass_interp,
    dims=['latitude', 'longitude'],
    coords={
        'latitude': grid_lat,
        'longitude': grid_lon
    }
)

# Assign CRS — WGS84 (EPSG:4326)
da = da.rio.set_spatial_dims(x_dim='longitude', y_dim='latitude')
da = da.rio.write_crs('EPSG:4326')

# Export to GeoTIFF
geotiff_filename = 'tephra_isomass.tif'
da.rio.to_raster(geotiff_filename)
print(f'GeoTIFF exported: {geotiff_filename}')
print(f'CRS: EPSG:4326 (WGS84)')
print(f'Grid size: {mass_interp.shape}')
print(f'Lon range: {grid_lon.min():.4f} to {grid_lon.max():.4f}')
print(f'Lat range: {grid_lat.min():.4f} to {grid_lat.max():.4f}')

The next cell is the previous plot simple isomass map code. 8/6/2026

In [ ]:
# # plot a simple isomass map
# dem.plot(cmap="gray_r", add_colorbar=False)

# # plot contours in kg/m^2 with solid lines
# cntr2 = plt.tricontour(lon, lat, mass, levels=[1e-6, 1,10,100,1000,10000] if not levels_solid else levels_solid,
#         colors=('k',),linestyles=('-',),linewidths=(1,))

# #plot contours in kg/m^2 with dashed lines
# cntr3 = plt.tricontour(lon, lat, mass, levels=[5,50,500] if not levels_dashed else levels_dashed,
#         colors=('k',),linestyles=('dashed',),linewidths=(0.5,))

# # a = plt.tricontour(easting, northing, mass)
# #label contours of solid lines
# plt.clabel(cntr2, fmt = '%2.1d', colors = 'w', fontsize=8) #contour line labels
# plt.clabel(cntr3, fmt = '%2.1d', colors = 'k', fontsize=8) #contour line labels

# poi_names = ['Parkdale', 'Rhododendron', 'Govt. Camp']

# plt.scatter(poi_lon, poi_lat, marker="*", s=50)

# for i in range(len(poi_lon)):
#     plt.annotate(
#         f"{poi_names[i]}\n{poi_mass.tolist()[i]:.2f} kg/m²",
#         (poi_lon.tolist()[i], poi_lat.tolist()[i]),
#         xytext=(5, 5), textcoords='offset points',
#         fontsize=7
#     )


# #plot the volcano location
# plt.plot(vent_longitude, vent_latitude, "r^")

# #plt.xlim((northing[mass > .1].min(), northing[mass > .1].max()))
# #plt.ylim((easting[mass > .1].min(), easting[mass > .1].max()))

# plt.xlim((lon.min(), lon.max()))
# plt.ylim((lat.min(), lat.max()))

# #map the map square
# plt.rcParams["figure.figsize"]=(8, 8)
# plt.xlabel("Longitude")
# plt.ylabel("Latitude")
# plt.title("Tephra Dispersion $kg/m^2$")
# plt.savefig(output_image)
# plt.grid(True)
# plt.show()

### Conclusions